This notebook should be used to compare TIN vs SVGP implimentaiton with:
1. The heatmaps for the RMSE, MAE and 95th percentile
2. Dredge estimates for the Desdeomnda shoal reach
3. Wilcoxon signed rank (or whatever autorank defaults to) based on RMSE or MAE

In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

import numpy as np
import xdem
import geoutils as gu

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

# TODO: use autorank to compare results between TIN and SVGP
import autorank

In [ ]:
WORK_DIR = Path('/mnt/teamgroup/AutoDBM/Public')
MBES = WORK_DIR / 'Multibeam_Validation_Points_For_Distribution'
SBES = WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution'
CRS = pd.read_csv(WORK_DIR / 'survey_CRS_list.csv')

mbes = list((WORK_DIR / 'Multibeam_Validation_Points_For_Distribution').glob('*.csv'))
sbes = list((WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution').glob('*.csv'))
crs_data = WORK_DIR / 'survey_CRS_list.csv'

geojson = Path(os.getcwd()).parent / 'shapefiles' / 'National_Channel_Framework_-6742069550782745925.geojson'
ncf = gpd.read_file(geojson)

In [ ]:
# Build a boundary catalog from SurveyJob layers, then match each survey to the best boundary by SurveyId
data_pairs = {}

def _norm(s):
    return "".join(ch for ch in str(s).upper() if ch.isalnum())

def _tokens(s):
    return {t for t in str(s).upper().replace("-", "_").split("_") if t}

# 1) Read all boundary candidates once
boundary_catalog = []
for gdb_file in WORK_DIR.glob("*/*.gdb"):
    try:
        sj = gpd.read_file(gdb_file, layer="SurveyJob")
    except Exception:
        continue

    if "SurveyId" not in sj.columns or sj.empty:
        continue

    # keep one record per SurveyId (first occurrence)
    for sid, grp in sj.groupby("SurveyId", dropna=True):
        boundary_catalog.append({
            "survey_id": str(sid),
            "survey_norm": _norm(sid),
            "survey_tokens": _tokens(sid),
            "gdb_file": gdb_file,
            "boundary": grp.copy()
        })

# 2) Match each survey from CRS table to best boundary
for _, items in pd.read_csv(crs_data).iterrows():
    survey = items["Survey"]

    sbes_match = [p for p in sbes if survey in str(p)]
    mbes_match = [p for p in mbes if survey in str(p)]
    if not sbes_match or not mbes_match:
        print(f"Missing SBES/MBES file for: {survey}")
        continue

    s_norm = _norm(survey)
    s_tokens = _tokens(survey)

    # exact normalized SurveyId match first
    exact = [c for c in boundary_catalog if c["survey_norm"] == s_norm]

    if exact:
        best = exact[0]
    else:
        # fallback: token overlap score
        scored = []
        for c in boundary_catalog:
            inter = len(s_tokens & c["survey_tokens"])
            denom = max(1, len(s_tokens))
            score = inter / denom
            scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1] if scored and scored[0][0] > 0 else None

    if best is None:
        print(f"No boundary found for: {survey}")
        continue

    boundary = best["boundary"]
    data_pairs[survey] = [
        sbes_match[0],
        mbes_match[0],
        boundary,
    ]

    print(f"{survey} -> {best['survey_id']} ({best['gdb_file'].name})")

In [ ]:
# survey = list(data_pairs.keys())[3]
survey = "CL_02_UDS_20260305_CS_A"

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

OUT_DIR = WORK_DIR / survey
os.makedirs(OUT_DIR, exist_ok=True)

sbes_x = sbes_gdf.X_ft
sbes_y = sbes_gdf.Y_ft
sbes_z = sbes_gdf.Elevation_ft * -1

mbes_x = mbes_gdf.X_ft
mbes_y = mbes_gdf.Y_ft
mbes_z = mbes_gdf.Elevation_ft * -1

fig, ax = plt.subplots(1,2, figsize=(10, 5))

# Histogram for sbes_z
ax[0].hist(sbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Elevation (ft)', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Elevation Distribution (n={len(sbes_z):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

# Histogram for mbes_z
ax[1].hist(mbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Elevation (ft)', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Elevation Distribution (n={len(mbes_z):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z):,}):")
print(f"  Mean: {np.mean(sbes_z):.2f} ft")
print(f"  Std: {np.std(sbes_z):.2f} ft")
print(f"  Min: {np.min(sbes_z):.2f} ft")
print(f"  Max: {np.max(sbes_z):.2f} ft")

print(f"\nMBES Elevation Statistics (n={len(mbes_z):,}):")
print(f"  Mean: {np.mean(mbes_z):.2f} ft")
print(f"  Std: {np.std(mbes_z):.2f} ft")
print(f"  Min: {np.min(mbes_z):.2f} ft")
print(f"  Max: {np.max(mbes_z):.2f} ft")

# Step 1: 
Generate 9x9 plots showing the 2, 40, 100 ft point spacing and 25, 100, 300 ft transect spacing for TIN and SVGP

In [ ]:
# load the reference MBES generated from all points with the TIN
mbes_dbm_path = Path(f'/mnt/teamgroup/AutoDBM/Public/{survey}/MBES_TIN_bathy.tif')
clipped_sbes_dbm_paths = list(Path(f'/mnt/teamgroup/AutoDBM/Public/{survey}').glob('clipped_SBES*'))
clipped_kriging_dbm_paths = list(Path(f'/mnt/teamgroup/AutoDBM/Public/{survey}').glob('isoKrige_clipped*pred*'))

tgt_ncf = ncf[ncf['channelareaidfk'].str.contains(f'{survey[:9]}')]

test_mbes_dbm = xdem.DEM(mbes_dbm_path)

test_sbes_dbm = xdem.DEM(f'/mnt/teamgroup/AutoDBM/Public/{survey}/clipped_SBES_25_4_TIN_bathy.tif')

In [ ]:
clipped_mbes_dbm = test_mbes_dbm.reproject(test_sbes_dbm)

if isinstance(test_sbes_dbm.data, np.ma.MaskedArray):
    sbes_valid_mask = ~np.ma.getmaskarray(test_sbes_dbm.data)
else:
    sbes_valid_mask = ~np.isnan(test_sbes_dbm.data)

mbes_data = clipped_mbes_dbm.data.copy()

if isinstance(mbes_data, np.ma.MaskedArray):
    mbes_masked = np.ma.masked_where(~sbes_valid_mask, mbes_data)
else:
    mbes_masked = np.where(sbes_valid_mask, mbes_data, np.nan)

masked_mbes_dbm = clipped_mbes_dbm.copy()
masked_mbes_dbm.data = mbes_masked

masked_mbes_dbm.plot(
    cmap="viridis",
    vmin=float(np.nanmin(mbes_masked)),
    vmax=float(np.nanmax(mbes_masked)),
    cbar_title="Elevation (ft, MLLW)",
)

## First, we create the plots for the TIN.
Bathy and Error

In [ ]:
price_per_cy = 9.29  # USD/CY dredged

dredge_cuts = {}

auth_depth = float(tgt_ncf["depthauthorized"].iloc[0])
elevation = masked_mbes_dbm.data

# Explicitly determine which cells have real data
if np.ma.is_masked(elevation):
    valid = ~np.ma.getmaskarray(elevation)
else:
    valid = np.isfinite(elevation)

elevation_filled = np.ma.filled(elevation, 0.0) if np.ma.is_masked(elevation) else elevation

# Compute cut = auth_depth + elevation (elevation is negative)
cut = auth_depth + elevation_filled
cut = np.where(cut > 0, cut, 0.0)
cut[~valid] = 0.0    # zero out anything without real survey data

px, py = abs(masked_mbes_dbm.transform.a), abs(masked_mbes_dbm.transform.e)
volume_ft3 = np.sum(cut) * px * py
volume_cy = volume_ft3 / 27.0

dredge_cuts['MBES'] = (volume_cy, volume_cy * price_per_cy, masked_mbes_dbm)
print(volume_cy)

In [ ]:
for path in clipped_sbes_dbm_paths:
    dbm = xdem.DEM(path).reproject(masked_mbes_dbm)

    elevation = dbm.data
    if elevation.mean() > 0:
        elevation = elevation * -1

    # Explicitly determine which cells have real data
    if np.ma.is_masked(elevation):
        valid = ~np.ma.getmaskarray(elevation)
    else:
        valid = np.isfinite(elevation)

    elevation_filled = np.ma.filled(elevation, 0.0) if np.ma.is_masked(elevation) else elevation

    # Compute cut = auth_depth + elevation (elevation is negative)
    cut = auth_depth + elevation_filled
    cut = np.where(cut > 0, cut, 0.0)
    cut[~valid] = 0.0    # zero out anything without real survey data

    px, py = abs(dbm.transform.a), abs(dbm.transform.e)
    volume_ft3 = np.sum(cut) * px * py
    volume_cy = volume_ft3 / 27.0

    dredge_cuts[path.stem[8:-10]] = (volume_cy, volume_cy * price_per_cy, dbm, masked_mbes_dbm + dbm) #masked_mbes_dbm - dbm

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
# sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# Compute a global elevation range across all DEMs so the shared colorbar is correct 
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, -dredge_cuts[dem_key][2].get_stats('max'))
        global_vmax = max(global_vmax, -dredge_cuts[dem_key][2].get_stats('min'))
del dem_key

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # Grab the pre-built DEM and plot the raster itself, using the shared global scale 
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = dredge_cuts[dem_key][2] * -1

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='viridis',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='viridis')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Elevation (ft MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN Prediction", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
# sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# Compute a global elevation range across all DEMs so the shared colorbar is correct 
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        global_vmin = min(global_vmin, -dredge_cuts[dem_key][3].get_stats('max'))
        global_vmax = max(global_vmax, -dredge_cuts[dem_key][3].get_stats('min'))
del dem_key

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # Grab the pre-built DEM and plot the raster itself, using the shared global scale 
        dem_key = f'SBES_{tspace * 25}_{pspace}'
        dem_this_tspace_pspace = dredge_cuts[dem_key][3] * -1

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Error (ft)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("TIN Error", fontsize=16, fontweight="bold", y=1.01)

plt.show()

## Second, we create the plots for the kriging
Bathy, Variance (?), and Error

**NOTE** Stochasticity in the inducing points seemse to play a role in accuracy of the bathy predictions

In [ ]:
kriging_bathy_path = clipped_kriging_dbm_paths[1]
print(kriging_bathy_path)

kriging_dbm = xdem.DEM(kriging_bathy_path)

clipped_kriging_dbm = kriging_dbm.reproject(masked_mbes_dbm)

if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
    mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
else:
    mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

kriging_data = clipped_kriging_dbm.data.copy()

if isinstance(kriging_data, np.ma.MaskedArray):
    kriging_masked = np.ma.masked_where(~mbes_valid_mask, kriging_data)
else:
    kriging_masked = np.where(mbes_valid_mask, kriging_data, np.nan)

masked_kriging_dbm = clipped_kriging_dbm.copy()
masked_kriging_dbm.data = kriging_masked

masked_kriging_dbm.plot(
    cmap="viridis",
    vmin=float(masked_kriging_dbm.get_stats("min")),
    vmax=float(masked_kriging_dbm.get_stats("max")),
    cbar_title="Elevation (ft, MLLW)",
)

In [ ]:
gp_unc_path = Path(str(kriging_bathy_path).replace('pred', 'var'))
print(gp_unc_path)

unc_obj = xdem.DEM(gp_unc_path)

clipped_gp_unc_dbm = unc_obj.reproject(masked_mbes_dbm)

if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
    mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
else:
    mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

gp_unc_data = clipped_gp_unc_dbm.data.copy()

if isinstance(gp_unc_data, np.ma.MaskedArray):
    gp_unc_masked = np.ma.masked_where(~mbes_valid_mask, gp_unc_data)
else:
    gp_unc_masked = np.where(mbes_valid_mask, gp_unc_data, np.nan)

masked_unc_obj = clipped_gp_unc_dbm.copy()
masked_unc_obj.data = gp_unc_masked

masked_unc_obj.plot(
    cmap="magma",
    vmin=float(masked_unc_obj.get_stats("min")),
    vmax=float(masked_unc_obj.get_stats("max")),
    cbar_title="Uncertainty",
)

In [ ]:
kriging_dredge_cuts = {}

for path in clipped_kriging_dbm_paths:
    # if path.stem == 'clipped_SVGP_depth_25_4':
    #     dbm = xdem.DEM(path).reproject(masked_mbes_dbm) * -1
    # else:
    #     dbm = xdem.DEM(path).reproject(masked_mbes_dbm)

    dbm = xdem.DEM(path).reproject(masked_mbes_dbm)

    if dbm.get_stats('mean') > 0:
        dbm = dbm * -1
    unc_dbm = xdem.DEM(str(path).replace('bathy', 'uncertainty')).reproject(masked_mbes_dbm)


    if isinstance(masked_mbes_dbm.data, np.ma.MaskedArray):
        mbes_valid_mask = ~np.ma.getmaskarray(masked_mbes_dbm.data)
    else:
        mbes_valid_mask = ~np.isnan(masked_mbes_dbm.data)

    gp_data = dbm.data.copy()

    if isinstance(gp_data, np.ma.MaskedArray):
        gp_masked = np.ma.masked_where(~mbes_valid_mask, gp_data)
    else:
        gp_masked = np.where(mbes_valid_mask, gp_data, np.nan)

    masked_gp_dbm = dbm.copy()
    masked_gp_dbm.data = gp_masked

    gp_unc_data = unc_dbm.data.copy()

    if isinstance(gp_unc_data, np.ma.MaskedArray):
        gp_unc_masked = np.ma.masked_where(~mbes_valid_mask, gp_unc_data)
    else:
        gp_unc_masked = np.where(mbes_valid_mask, gp_unc_data, np.nan)

    masked_unc_obj = unc_dbm.copy()
    masked_unc_obj.data = gp_unc_masked

    # raw_data = dbm.data
    raw_data = gp_masked
    if isinstance(raw_data, np.ma.MaskedArray):
        nodata_mask = np.ma.getmaskarray(raw_data)
        elevation = raw_data.filled(np.nan)
    else:
        elevation = np.asarray(raw_data, dtype=float)
        nodata_mask = np.isnan(elevation)

    # raw_unc = unc_dbm.data
    raw_unc = gp_unc_masked
    if isinstance(raw_unc, np.ma.MaskedArray):
        nodata_mask = np.ma.getmaskarray(raw_unc)
        unc = raw_unc.filled(np.nan)
    else:
        unc = np.asarray(raw_unc, dtype=float)
        nodata_mask = np.isnan(unc)

    # Compute cut = auth_depth + elevation (elevation is negative)
    cut = auth_depth + elevation
    cut = np.where(cut > 0, cut, 0.0)
    cut[nodata_mask] = 0.0

    cut_up = cut + (1.96*unc)
    cut_up = np.where(cut_up > 0, cut_up, 0.0)
    cut_up[nodata_mask] = 0.0

    cut_lo = cut - (1.96*unc)
    cut_lo = np.where(cut_lo > 0, cut_lo, 0.0)
    cut_lo[nodata_mask] = 0.0

    px, py = abs(masked_gp_dbm.transform.a), abs(masked_gp_dbm.transform.e)
    volume_ft3 = np.sum(cut) * px * py
    volume_ft3_up = np.sum(cut_up) * px * py
    volume_ft3_lo = np.sum(cut_lo) * px * py
    volume_cy = volume_ft3 / 27.0
    volume_cy_up = volume_ft3_up / 27.0
    volume_cy_lo = volume_ft3_lo / 27.0

    kriging_dredge_cuts[path.stem[9:]] = (volume_cy, volume_cy * price_per_cy,
                                       volume_cy_up, volume_cy_up * price_per_cy,
                                       volume_cy_lo, volume_cy_lo * price_per_cy,
                                       masked_gp_dbm, masked_unc_obj,
                                       masked_mbes_dbm - dbm)

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# Compute a global elevation range across all DEMs so the shared colorbar is correct 
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'clipped_SBES_{tspace*25}_{pspace}_pred'
        global_vmin = min(global_vmin, kriging_dredge_cuts[dem_key][6].get_stats('min'))
        global_vmax = max(global_vmax, kriging_dredge_cuts[dem_key][6].get_stats('max'))
del dem_key

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        # Grab the pre-built DEM and plot the raster itself, using the shared global scale
        dem_key = f'clipped_SBES_{tspace*25}_{pspace}_pred'
        dem_this_tspace_pspace = kriging_dredge_cuts[dem_key][6]

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='viridis',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='viridis')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Elevation (ft MLLW)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("Kriging Prediction", fontsize=16, fontweight="bold", y=1.01)

plt.show()

In [ ]:
transects_to_plot = list(range(380, 490))

transect_spacings_to_plot = [12, 4, 1]  # the original transects I created are 25 ft apart, so we can downsample to achieve the other transect spacings.
point_spacings_to_plot = [4, 20, 100]  # the original points I created are 2 ft apart, so we can downsample to achieve the desired point spacing

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

sbes_gdf = sbes_gdf[sbes_gdf['Transect_ID'].isin(transects_to_plot)].reset_index(drop=True)
sbes_gdf['Elevation_ft'] = np.multiply(sbes_gdf['Elevation_ft'], -1)

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

num_transects = max(sbes_gdf['Transect_ID']) - min(sbes_gdf['Transect_ID'])  # how many transects are available in the singlebeam survey?

# Compute a global elevation range across all DEMs so the shared colorbar is correct 
global_vmin = np.inf
global_vmax = -np.inf
for tspace in transect_spacings_to_plot:
    for pspace in point_spacings_to_plot:
        dem_key = f'clipped_SBES_{tspace*25}_{pspace}_pred'
        global_vmin = min(global_vmin, -kriging_dredge_cuts[dem_key][8].get_stats('max'))
        global_vmax = max(global_vmax, -kriging_dredge_cuts[dem_key][8].get_stats('min'))
del dem_key

fig, axes = plt.subplots(len(transect_spacings_to_plot), len(point_spacings_to_plot), figsize=(6.5, 6.5), dpi=300)
axes = np.atleast_2d(axes)  # ensures axes is always 2D, even if one dimension is 1

for itspace, tspace in enumerate(transect_spacings_to_plot):  # loop through the different transect spacings

    transects_to_use = list(np.arange(min(sbes_gdf['Transect_ID']), min(sbes_gdf['Transect_ID']) + num_transects + 1, tspace))  # transect indices that we will use to achieve the desired spacing

    # Downsample the synthetic singlebeam data to only contain the desired transect spacing.
    transect_id_temp = list(sbes_gdf['Transect_ID'])  # transect ID for each row of the singlebeam dataframe
    use_row = [(id in transects_to_use) for id in transect_id_temp]
    singlebeam_df_this_tspace = sbes_gdf[use_row].reset_index(drop=True)
    del (use_row, transect_id_temp)

    # Remove any points used to create this set of transects from the multibeam validation data.
    multibeam_df_this_tspace = mbes_gdf[~mbes_gdf['Used_to_create_transect'].isin(transects_to_use)].reset_index(drop=True)

    for ipspace, pspace in enumerate(point_spacings_to_plot):  # loop through the different point spacings

        along_length_temp = np.array(singlebeam_df_this_tspace['Length_along_transect_ft'])
        along_length_temp = np.round(along_length_temp)  # some of them are decimals??? I have no clue why Arc did that.
        use_row = [(np.mod(thislength, pspace) == 0) for thislength in along_length_temp]
        singlebeam_this_tspace_pspace = singlebeam_df_this_tspace[use_row].reset_index(drop=True)  # this is the final set of points that we will use to create the TIN.
        del (along_length_temp, use_row)

        dem_key = f'clipped_SBES_{tspace*25}_{pspace}_pred'
        dem_this_tspace_pspace = kriging_dredge_cuts[dem_key][8] * -1

        dem_this_tspace_pspace.plot(
            ax=axes[itspace, ipspace],
            cmap='PuOr',
            add_cbar=False,
            vmin=global_vmin,
            vmax=global_vmax,
        )

        # Get the indices of the first three transects for plotting
        these_transects = list(singlebeam_this_tspace_pspace['Transect_ID'])
        these_transects = sorted(list(set(these_transects)))
        transects_to_plot_local = these_transects[:3]
        df_for_plot = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'].isin(transects_to_plot_local)].reset_index(drop=True)

        for transect in transects_to_plot_local:
            df_filt = singlebeam_this_tspace_pspace[singlebeam_this_tspace_pspace['Transect_ID'] == transect].reset_index(drop=True)
            axes[itspace, ipspace].plot(df_filt['X_ft'], df_filt['Y_ft'], linewidth=0.2, c='black', zorder=2)
        axes[itspace, ipspace].scatter(df_for_plot['X_ft'], df_for_plot['Y_ft'], s=0.4, c='black', zorder=3)
        axes[itspace, ipspace].set_aspect('equal')
        axes[itspace, ipspace].set_xticklabels([])
        axes[itspace, ipspace].set_yticklabels([])
        axes[itspace, ipspace].tick_params(left=False, bottom=False)

        if ipspace == 0:
            axes[itspace, ipspace].set_ylabel(str(tspace * 25) + ' ft', fontsize=11)

        if itspace == 2:
            axes[itspace, ipspace].set_xlabel(str(pspace) + ' ft', fontsize=11)

# Layout: run tight_layout first, THEN reserve margin and place sup-labels, so they aren't overridden 
plt.tight_layout()
fig.subplots_adjust(right=0.83, bottom=0.20, wspace=0.08, hspace=0.08)
fig.supxlabel('Point spacing on transect', fontsize=15, fontweight='bold', y=0.12, x=0.45)
fig.canvas.draw()
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
grid_bottom = min(ax.get_position().y0 for ax in axes.flat)
grid_top = max(ax.get_position().y1 for ax in axes.flat)
fig.text(grid_left - 0.07, (grid_bottom + grid_top) / 2, 'Transect spacing', fontsize=15, fontweight='bold', rotation='vertical', va='center', ha='center')

# Use an explicit shared mappable so the colorbar is guaranteed to use the same global limits.
shared_mappable = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=global_vmin, vmax=global_vmax), cmap='PuOr')
shared_mappable.set_array([])

cbar_ax = fig.add_axes([grid_right + 0.02, grid_bottom, 0.03, grid_top - grid_bottom])
cbar = fig.colorbar(shared_mappable, cax=cbar_ax)
cbar.set_label('Error (ft)', fontsize=13)
cbar.ax.tick_params(rotation=90)

fig.suptitle("Gaussian Process Error", fontsize=16, fontweight="bold", y=1.01)

plt.show()

## Next, extract the RMSE, MAE and 95th Percentile of |Error| for the TIN and SVGP
- Will plot the error heatmaps for each of the three, then try to subtract the SVGP from the TIN to show the differences

In [ ]:
survey_mbes = {}
survey_sbes = {}

for survey in list(data_pairs.keys()):
    survey_mbes[survey] = list(WORK_DIR.glob(f'{survey}/*MBES*.tif'))
    survey_sbes[survey] = list(WORK_DIR.glob(f'{survey}/SBES*TIN*.tif'))

    survey_sbes[survey] += list(WORK_DIR.glob(f'{survey}/isoKrige_SBES*pred.tif'))

In [ ]:
# NOTE: This is for the TIN

tin_records = []
for survey in data_pairs:
    mbes_dbm = xdem.DEM(survey_mbes[survey][0])

    for tif in survey_sbes[survey]:
        if tif.stem == 'SBES_TIN_bathy' or 'SVGP' in  tif.stem:
            continue
        ddbm = mbes_dbm - xdem.DEM(tif).reproject(mbes_dbm)
        
        # TODO: edit this to work for the SVGP paths too
        # NOTE: the track spacing and sounding spacing are in differnet postions
        if 'TIN' in tif.stem:
            ts, ss = tif.stem[5:-10].split('_')
            interp_method = 'TIN'
        else:
            ts, ss = tif.stem[11:].split('_')
            interp_method = 'SVGP'

        tin_records.append({
            'Track_Spacing': ts,
            'Sounding_Spacing': ss,
            'MAE': np.mean(np.abs(ddbm)),
            'RMSE': ddbm.get_stats('rmse'),
            # 'NMAD': ddbm.get_stats('nmad'),
            '95_Percentile': np.percentile(np.abs(ddbm), 95),
        })

tin_df = pd.DataFrame(tin_records)
tin_mean_agg_df = tin_df.groupby(['Track_Spacing', 'Sounding_Spacing']).mean().reset_index()
# med_agg_df = df.groupby(['Track_Spacing', 'Sounding_Spacing']).median().reset_index()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

tin_mean_agg_df['Track_Spacing'] = tin_mean_agg_df['Track_Spacing'].astype(float)
tin_mean_agg_df['Sounding_Spacing'] = tin_mean_agg_df['Sounding_Spacing'].astype(float)

# (column, title, cmap, colorbar label, vmin)
metrics = [
    ('MAE', 'Global MAE', 'Blues', 'MAE (ft)', 0),
    ('RMSE', 'Global RMSE', 'Oranges', 'RMSE (ft)', 0),
    ('95_Percentile', '95th Percentile of |Error|', 'Purples', '95th percentile (ft)', None),
]

# constrained layout handles colorbars properly (tight_layout doesn't)
fig, axes = plt.subplots(1, 3, figsize=(12, 5), dpi=500, layout='constrained')
fig.get_layout_engine().set(w_pad=0.15, h_pad=0.1, wspace=0.05)

for ax, (col, title, cmap, cbar_label, vmin) in zip(axes, metrics):
    grid = (tin_mean_agg_df.pivot(index='Track_Spacing',
                                  columns='Sounding_Spacing',
                                  values=col)
            .sort_index(ascending=False)   # transect spacing: 500 at top
            .sort_index(axis=1))           # point spacing: 4 -> 100

    im = ax.imshow(grid.values, cmap=cmap, aspect='auto', vmin=vmin)
    norm = im.norm

    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            val = grid.values[r, c]
            if not np.isnan(val):
                color = 'white' if norm(val) > 0.75 else 'black'
                ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                        fontsize=8, color=color)

    ax.set_xticks(range(grid.shape[1]))
    ax.set_xticklabels([f'{int(x)}' for x in grid.columns], fontsize=9)
    ax.set_yticks(range(grid.shape[0]))
    ax.set_yticklabels([f'{int(y)}' for y in grid.index],
                       rotation=90, va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Point spacing on transect (ft)', fontweight='bold',
                  fontsize=10, labelpad=2)
    ax.set_ylabel('Transect spacing (ft)', fontweight='bold', fontsize=10)

    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', location='bottom',
                        shrink=0.7, aspect=18, pad=0.04)
    cbar.set_label(cbar_label, fontweight='bold', fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    cbar.locator = MaxNLocator(nbins=3)
    cbar.update_ticks()

# black frame around the whole figure
fig.patch.set_edgecolor('black')
fig.patch.set_linewidth(1.5)

# plt.savefig('error_heatmaps.png', dpi=500, edgecolor=fig.get_edgecolor())
plt.show()

In [ ]:
tin_mean_agg_df['Track_Spacing'] = tin_mean_agg_df['Track_Spacing'].astype(float)
tin_mean_agg_df['Sounding_Spacing'] = tin_mean_agg_df['Sounding_Spacing'].astype(float)

metrics = [
    ('MAE', 'Global MAE', 'Blues', 'MAE (ft)'),
    ('RMSE', 'Global RMSE', 'Oranges', 'RMSE (ft)'),
    ('95_Percentile', '95th Percentile of |Error|', 'Purples', '95th percentile (ft)'),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 5), dpi=500)

for ax, (col, title, cmap, cbar_label) in zip(axes, metrics):
    # rows = transect (track) spacing descending, cols = point (sounding) spacing ascending
    grid = tin_mean_agg_df.pivot(index='Track_Spacing',
                        columns='Sounding_Spacing',
                        values=col).sort_index(ascending=False)

    im = ax.imshow(grid.values, cmap=cmap, aspect='auto')

    norm = im.norm  # maps data values to 0–1
    # annotate each cell
    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            val = grid.values[r, c]
            if not np.isnan(val):
                color = 'white' if norm(val) > 0.65 else 'black'
                ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                        fontsize=12, color=color)

    ax.set_xticks(range(grid.shape[1]))
    ax.set_xticklabels([f'{int(x)}' for x in grid.columns], fontsize=12)
    ax.set_yticks(range(grid.shape[0]))
    ax.set_yticklabels([f'{int(y)}' for y in grid.index], rotation='vertical', fontsize =12)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel('Point spacing on transect (ft)', fontweight='bold', fontsize =12)
    ax.set_ylabel('Transect spacing (ft)', fontweight='bold', fontsize =12)

    cbar = fig.colorbar(im, ax=ax, orientation='horizontal',
                        pad=0.16, fraction=0.05)
    cbar.set_label(cbar_label, fontweight='bold', fontsize=12)

plt.tight_layout()
# plt.savefig('error_heatmaps.png', dpi=500, bbox_inches='tight')
plt.show()